In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from sklearn.cluster import Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.decomposition import PCA

In [42]:
df=pd.read_csv('../data/new_amazon_sales.csv')

In [43]:
df.head()

,Sales,Profit,Quantity,Discount,Total_Orders,Avg_Order_Value
0,5563.560,-362.8825,30,0.090909,5,1112.712000
1,1056.390,277.3824,41,0.080000,9,117.376667
2,1790.512,435.8274,36,0.016667,4,447.628000
3,5086.935,857.8033,64,0.063889,6,847.822500
4,886.156,129.3465,13,0.066667,3,295.385333


In [44]:
df.describe()


,Sales,Profit,Quantity,Discount,Total_Orders,Avg_Order_Value
count,793.000000,793.000000,793.000000,793.000000,793.000000,793.000000
mean,2896.848500,361.156396,47.759142,0.157482,6.316520,460.147734
std,2628.670117,894.261812,24.842915,0.089071,2.550885,433.400951
min,4.833000,-6626.389500,2.000000,0.000000,1.000000,2.416500
25%,1146.050000,36.613100,30.000000,0.090909,5.000000,213.255333
50%,2256.394000,227.833800,44.000000,0.150000,6.000000,362.503250
75%,3785.276000,560.007800,63.000000,0.211111,8.000000,550.377000
max,25043.050000,8981.323900,150.000000,0.700000,17.000000,5008.610000


In [55]:
X = pd.DataFrame(df, columns=df.columns)

In [56]:
X.head()

,Sales,Profit,Quantity,Discount,Total_Orders,Avg_Order_Value
0,5563.560,-362.8825,30,0.090909,5,1112.712000
1,1056.390,277.3824,41,0.080000,9,117.376667
2,1790.512,435.8274,36,0.016667,4,447.628000
3,5086.935,857.8033,64,0.063889,6,847.822500
4,886.156,129.3465,13,0.066667,3,295.385333


In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_model = X_scaled

In [60]:
for k in range(2,10):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_model)

    score = silhouette_score(X_model, labels)
    print(k, score)

2 0.4914910486934111
3 0.25310998667641604
4 0.2720945540848886
5 0.2511474589675609
6 0.2509324107488908
7 0.2220910586152318
8 0.2206078754899505
9 0.20402002475363404


In [63]:
print(type(X_model))

<class 'numpy.ndarray'>


In [69]:
kmeans = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=20,
    random_state=42
)
labels = kmeans.fit_predict(X_model)

df["cluster"] = labels

In [70]:
df["cluster"].value_counts()

cluster
0    586
2    195
1     12
Name: count, dtype: int64

In [71]:
df.groupby("cluster").mean()

,Sales,Profit,Quantity,Discount,Total_Orders,Avg_Order_Value
cluster,,,,,,
0,1822.115944,72.094667,40.348123,0.169213,5.750853,324.380340
1,13668.983500,4019.612492,49.000000,0.121322,5.583333,2627.111568
2,5463.656999,1004.688189,69.953846,0.124452,8.061538,734.794538


In [80]:
models = {
"KMeans": KMeans()
}


for name, model in models.items():
    
    # Fit model
    if name == "Gaussian Mixture":
        labels = model.fit_predict(X_scaled)
    else:
        labels = model.fit_predict(X_scaled)

    print(name)
    
    if len(set(labels)) > 1:
        
        silhouette = silhouette_score(X_scaled, labels)
        db_index = davies_bouldin_score(X_scaled, labels)
        ch_score = calinski_harabasz_score(X_scaled, labels)

        print("Model performance")
        print("- Silhouette Score: {:.4f}".format(silhouette))
        print("- Davies-Bouldin Index: {:.4f}".format(db_index))
        print("- Calinski-Harabasz Score: {:.4f}".format(ch_score))
    
    else:
        print("Model produced only one cluster. Metrics cannot be computed.")
    
    print("="*35)
    print("\n")

KMeans
Model performance
- Silhouette Score: 0.2186
- Davies-Bouldin Index: 1.1340
- Calinski-Harabasz Score: 239.0929




In [78]:
import pickle

# save scaler
with open("../files/customer_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# save clustering model
with open("../files/customer_kmeans.pkl", "wb") as f:
    pickle.dump(kmeans, f)